In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# -----------------------------
# Dataset
# -----------------------------
dataset = load_dataset("stanfordnlp/imdb")
texts = dataset["train"]["text"][:5000]

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 128

enc = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="pt"
)

class GPTDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        x = self.ids[idx]
        return {
            "input_ids": x[:-1],
            "labels": x[1:]
        }

train_loader = DataLoader(
    GPTDataset(enc["input_ids"]),
    batch_size=8,
    shuffle=True
)

# -----------------------------
# GPT Model
# -----------------------------
class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8,
                 layers=6, max_len=128, dropout=0.1):
        super().__init__()

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, ids):
        B, T = ids.shape

        pos = torch.arange(T, device=ids.device).unsqueeze(0)

        x = self.token_embed(ids) + self.pos_embed(pos)

        mask = torch.triu(
            torch.ones(T, T, device=ids.device),
            diagonal=1
        ).bool()

        x = self.transformer(
            x,
            mask=mask
        )

        x = self.ln(x)

        return self.fc(x)

model = GPTModel(tokenizer.vocab_size, max_len=MAX_LEN).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

EPOCHS = 5

print("Training...")

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(ids)

        loss = criterion(
            logits.reshape(-1, tokenizer.vocab_size),
            labels.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(train_loader):.4f}")

torch.save(model.state_dict(), "mini_gpt_imdb.pth")
print("Model saved.")

# -----------------------------
# Generation
# -----------------------------
model.eval()

def generate(prompt, max_new_tokens=40, temperature=0.8, top_k=40):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            if ids.size(1) >= MAX_LEN-1:
                break

            logits = model(ids)
            logits = logits[:, -1, :] / temperature

            values, indices = torch.topk(logits, top_k)

            probs = torch.softmax(values, dim=-1)

            next_id = indices.gather(
                -1,
                torch.multinomial(probs, 1)
            )

            ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(ids[0], skip_special_tokens=True)

while True:
    text = input("\nPrompt: ")

    if text.lower() == "exit":
        break

    print("\nGenerated:\n")
    print(generate(text))

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using: cuda


Training...
Epoch 1: 6.6318
Epoch 2: 5.8146
Epoch 3: 5.4639
Epoch 4: 5.1656
Epoch 5: 4.8912
Model saved.
